In [2]:
from Tokenizer import Tokenizer
import re

with open('data.txt') as file:
    data = file.read()

In [3]:
#tokenize data
result = re.split(r'([,.:;?_!"()\']|--|\s)', data)
result = [i for i in result if i.strip()]
words = sorted(set(result))
words.extend(["<|unknown|>", "<|endoftext|>"])
token_id = {j:i for i,j in enumerate(words)}
tokenizerObj = Tokenizer(token_id)
encodedData = tokenizerObj.encode(data)

In [4]:
test = encodedData[0:10]
for end in range(1,len(test)):
    input = test[0:end]
    target = [test[end]]
    print("input ->" , tokenizerObj.decode(input), " | output-> ",tokenizerObj.decode(target))

input -> I  | output->  HAD
input -> I HAD  | output->  always
input -> I HAD always  | output->  thought
input -> I HAD always thought  | output->  Jack
input -> I HAD always thought Jack  | output->  Gisburn
input -> I HAD always thought Jack Gisburn  | output->  rather
input -> I HAD always thought Jack Gisburn rather  | output->  a
input -> I HAD always thought Jack Gisburn rather a  | output->  cheap
input -> I HAD always thought Jack Gisburn rather a cheap  | output->  genius


In [5]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        #while initializing itself, we are loading the data which will be used in __getitem__ function
        self.input_ids = []
        self.target_ids = []
        self.token_ids = tokenizer.encode(txt)                         #A
        for i in range(0, len(self.token_ids) - max_length, stride):   #B
            input_chunk = self.token_ids[i:i + max_length]
            target_chunk = self.token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    #mandatory method which Dataset Abstract class tells to implement
    def __len__(self):                                            #C
        return len(self.input_ids)
    #mandatory method which Dataset Abstract class tells to implement. Dataloader uses this to get the data from index. 
    # data loader takes care of indexes only for batching, shuffeling etc. Thats why we are inheriting Dataset parent abstract class
    def __getitem__(self, idx):                                   #D
        return self.input_ids[idx], self.target_ids[idx]

In [6]:
import tiktoken

def dataLoadingExperiment(maxlength, stride, batchSize, dropLast):
    print("-------------Experiment-------------------")
    print("maxlength=",maxlength, "stride=",stride, "batchSize=",batchSize, "dropLast=",dropLast)
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(data,tokenizer=tokenizer,max_length=maxlength,stride=stride)
    dataloader = DataLoader(dataset=dataset, shuffle=True, batch_size=batchSize, drop_last=dropLast, num_workers=0)

    for batchIdx, (x,y) in enumerate(dataloader):
        print("Batch ",batchIdx)
        print("no.of items in batch ",len(x))
        print("length of 0th tensor ",len(x[0]))
        break

    print("length of raw data text ", len(data))
    tokenLength = len(dataloader.dataset.token_ids)
    print("after converting to tokens and assigning ids ",tokenLength)
    sampleSizeCalculated =  int((tokenLength-maxlength)/stride)+1
    print("no.of inputs after sampling ",len(dataloader.dataset.input_ids), ". calculated as int((tokenLength-maxlength)/stride)+1= ", sampleSizeCalculated)
    print("batch size (no.of inputs in each batch) is ",batchSize)
    numberOfBatches = sampleSizeCalculated/batchSize
    print("numberOfBatches Calculates as sampleSizeCalculated/batchSize", numberOfBatches)
    if dropLast:
        print("dropLast is True. So, no.of batches is int(",numberOfBatches,")-", int(numberOfBatches))
    else:
        print("dropLast is False. So, no.of batches is round(",numberOfBatches,")-", round(numberOfBatches))
    print("number of batches from dataloader- ", len(dataloader))

In [7]:
dataLoadingExperiment(maxlength=256, stride=128, batchSize=4, dropLast=True)

-------------Experiment-------------------
maxlength= 256 stride= 128 batchSize= 4 dropLast= True
Batch  0
no.of items in batch  4
length of 0th tensor  256
length of raw data text  20479
after converting to tokens and assigning ids  5145
no.of inputs after sampling  39 . calculated as int((tokenLength-maxlength)/stride)+1=  39
batch size (no.of inputs in each batch) is  4
numberOfBatches Calculates as sampleSizeCalculated/batchSize 9.75
dropLast is True. So, no.of batches is int( 9.75 )- 9
number of batches from dataloader-  9


In [8]:
dataLoadingExperiment(maxlength=128, stride=128, batchSize=2, dropLast=True)

-------------Experiment-------------------
maxlength= 128 stride= 128 batchSize= 2 dropLast= True
Batch  0
no.of items in batch  2
length of 0th tensor  128
length of raw data text  20479
after converting to tokens and assigning ids  5145
no.of inputs after sampling  40 . calculated as int((tokenLength-maxlength)/stride)+1=  40
batch size (no.of inputs in each batch) is  2
numberOfBatches Calculates as sampleSizeCalculated/batchSize 20.0
dropLast is True. So, no.of batches is int( 20.0 )- 20
number of batches from dataloader-  20


In [9]:
dataLoadingExperiment(maxlength=128, stride=64, batchSize=2, dropLast=False)

-------------Experiment-------------------
maxlength= 128 stride= 64 batchSize= 2 dropLast= False
Batch  0
no.of items in batch  2
length of 0th tensor  128
length of raw data text  20479
after converting to tokens and assigning ids  5145
no.of inputs after sampling  79 . calculated as int((tokenLength-maxlength)/stride)+1=  79
batch size (no.of inputs in each batch) is  2
numberOfBatches Calculates as sampleSizeCalculated/batchSize 39.5
dropLast is False. So, no.of batches is round( 39.5 )- 40
number of batches from dataloader-  40


In [15]:
tokenizer = tiktoken.get_encoding("gpt2")
dataset = GPTDatasetV1(data,tokenizer=tokenizer,max_length=256,stride=128)
dataloader = DataLoader(dataset=dataset, shuffle=True, batch_size=4, drop_last=True, num_workers=0)
torch.manual_seed(123)
vocab_size = 50257
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
context_length = 256
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape)

torch.Size([256, 256])


In [13]:
for batch in dataloader:
    x, y = batch

    token_embeddings = token_embedding_layer(x)
    pos_embeddings = pos_embedding_layer(torch.arange(256))

    input_embeddings = token_embeddings + pos_embeddings

    break

In [14]:
print(input_embeddings.shape)

torch.Size([4, 256, 256])
